In [ ]:
import requests
import pandas as pd
import time

In [ ]:
# 1. Initialzation
api_key = "d0008fd913ba7facf65ddf3956c811e4"
base_url = "https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/0000/var/2194/th/{year_id}/key/{key}"
target_turvar = "1550" # ID for PDRB

# Generate Year IDs from 110 (2010) to 125 (2025)
year_ids = [str(i) for i in range(110, 126)]

# Provinsi Mapping
prov_map = {
    '11': 'Aceh', '12': 'Sumatera Utara', '13': 'Sumatera Barat', '14': 'Riau', '15': 'Jambi',
    '16': 'Sumatera Selatan', '17': 'Bengkulu', '18': 'Lampung', '19': 'Kep. Bangka Belitung',
    '21': 'Kep. Riau', '31': 'DKI Jakarta', '32': 'Jawa Barat', '33': 'Jawa Tengah',
    '34': 'DI Yogyakarta', '35': 'Jawa Timur', '36': 'Banten', '51': 'Bali',
    '52': 'Nusa Tenggara Barat', '53': 'Nusa Tenggara Timur', '61': 'Kalimantan Barat',
    '62': 'Kalimantan Tengah', '63': 'Kalimantan Selatan', '64': 'Kalimantan Timur',
    '65': 'Kalimantan Utara', '71': 'Sulawesi Utara', '72': 'Sulawesi Tengah',
    '73': 'Sulawesi Selatan', '74': 'Sulawesi Tenggara', '75': 'Gorontalo',
    '76': 'Sulawesi Barat', '81': 'Maluku', '82': 'Maluku Utara', '91': 'Papua Barat',
    '92': 'Papua Barat Daya', '94': 'Papua', '95': 'Papua Selatan', '96': 'Papua Tengah',
    '97': 'Papua Pegunungan', 
}

In [ ]:
print("Starting Data Collection...")

all_data = []

# Pastikan fungsi mapping provinsi sudah didefinisikan sebelumnya
def get_prov_from_id(kab_id):
    kode_prov = str(kab_id)[:2]
    return prov_map.get(kode_prov, f"Unknown ({kode_prov})")

for y_id in year_ids:
    url = base_url.format(year_id=y_id, key=api_key)
    
    try:
        response = requests.get(url)
        json_data = response.json()
        
        if json_data['status'] == 'OK':
            # 1. Create Mappings
            kab_map = {str(item['val']): item['label'] for item in json_data['vervar']}
            thn_map = {str(item['val']): item['label'] for item in json_data['tahun']}
            
            # 2. Extract Data Content
            datacontent = json_data['datacontent']
            
            # 3. Strict Matching Loop
            # Kita loop melalui datacontent, lalu cari kombinasi ID yang pas
            for full_id, value in datacontent.items():
                for k_id, k_name in kab_map.items():
                    for t_id, t_name in thn_map.items():
                        
                        # Konstruksi ID sesuai hasil reverse-engineering kita:
                        # [Domain/Kab ID] + [Var 2194] + [Turvar 1550] + [Thn ID] + [Turtahun 0]
                        # Catatan: Jika urutan di JSON kamu terbalik, sesuaikan f-string ini
                        expected_id = f"{k_id}21941550{t_id}0" 
                        
                        if full_id == expected_id:
                            all_data.append({
                                'Provinsi': get_prov_from_id(k_id),
                                'Kabupaten': k_name,
                                'Tahun': t_name,
                                'PDRB': float(value) if value else 0,
                                'raw_id': full_id # Opsional: simpan untuk audit
                            })
            
            print(f"Successfully collected data for Year ID: {y_id}")
            
        else:
            print(f"Skipping Year ID {y_id}: {json_data['message']}")
            
    except Exception as e:
        print(f"Error on Year ID {y_id}: {e}")
    
    # Delay 1 detik agar tidak terkena rate limit API BPS
    time.sleep(1)

In [ ]:
# 2. Combine into 1 DataFrame
df_final = pd.DataFrame(all_data)

# 3. Final Cleaning
#df_final = df_final[df_final['Kabupaten'] != "Unknown"]

print("\nFinal Data Collection Complete!")
display(df_final.head())

In [ ]:
# 1. Force the Value column to be numeric (converts non-numbers to NaN)
df_final['PDRB'] = pd.to_numeric(df_final['PDRB'], errors='coerce')

# 2. Filter out both Python Nulls (NaN) AND the "Unknown" strings in one line
df_final_clean = df_final[
    (df_final['Kabupaten'] != "Unknown") & 
    (df_final['Provinsi'] != "Unknown") & 
    (df_final['PDRB'].notnull())
].copy()

# Define the new order
new_order = ['Provinsi', 'Kabupaten', 'Tahun', 'PDRB']

# Apply the order to your clean dataframe
df_final_clean = df_final_clean[new_order]

# Final verification print
print(f"Cleanup complete. {len(df_final_clean)} valid rows ready for PostgreSQL.")
display(df_final_clean.head())

***

## 🚴🏻 Push the data into PostgreSQL using ✨SQLAlchemy✨

In [ ]:
from sqlalchemy import create_engine

# Replace with your DBeaver/PostgreSQL credentials
user = "postgres"
password = "190420"
host = "localhost"
port = "5432"
db_name = "bps_analysis" # Make sure this DB exists in DBeaver!

# Create the connection string
connection_string = f"postgresql://{user}:{password}@{host}:{port}/{db_name}"
engine = create_engine(connection_string)

print("Engine created. Ready to push data!")

In [ ]:
# 1. Define the table name
table_name = "pdrb_regional_2010_2025"

# 2. Push to SQL
# if_exists='replace' will create the table if it doesn't exist 
# or overwrite it if you run the script again.
df_final_clean.to_sql(table_name, con=engine, if_exists='replace', index=False)

print(f"Success! {len(df_final_clean)} rows pushed to table: {table_name}")